[Home](../../README.md)

### Data Wrangling

This is a demonstration of data wrangling using [Pandas](https://pandas.pydata.org/) the library for data analysis and manipulation.

This Jupyter Notepad demonstrates different processes you can apply to your data to prepare it for feature engineering and model training. For this demonstration we will wrangle the diabetes data set you previewed in the last Jupyter Notebook.

> [!Note]
> None of these processes are destructive to the source CSV as long as you save the modified data to a new CSV.

#### Load the required dependencies

In [1]:
# Import frameworks
import pandas as pd

####  Store the data as a local variable

The raw WCA CSV files are loaded into Pandas DataFrames so the data can be inspected, cleaned, and prepared for machine learning. Each dataset is stored as a variable which allows the full data to be held for preprocessing. I dropped unneccessary columns to reduce noise and memory usage, and I filtered the results dataset to include only valid 3×3×3 solves with non-zero averages to fit the business/ml problem. Doing this ensures that the data used for modelling is clean, relevant, and consistent. The additional datasets such as persons, countries, and competitions are also loaded and simplified so they can be merged or referenced during feature engineering and analysis.

In [2]:
results = pd.read_csv("/workspaces/2026SE_MLOPSTask_Michael.L/wca_export/csv/WCA_export_results.csv")
results = results.drop(columns=["round_type_id", "id", "format_id", "regional_single_record", "regional_average_record"])
results = results[results["event_id"] == "333"]
results = results[results["average"] > 0]

results_all_events = pd.read_csv("/workspaces/2026SE_MLOPSTask_Michael.L/wca_export/csv/WCA_export_results.csv")
results_all_events = results_all_events.drop(columns=["round_type_id", "id", "format_id", "regional_single_record", "regional_average_record", "pos", "best", "average", "person_name", "person_country_id"])

persons = pd.read_csv("/workspaces/2026SE_MLOPSTask_Michael.L/wca_export/csv/WCA_export_persons.csv")
persons = persons.drop(columns=["sub_id"])

countries = pd.read_csv("/workspaces/2026SE_MLOPSTask_Michael.L/wca_export/csv/WCA_export_countries.csv")
countries = countries.drop(columns=["iso2", "name"])

competitions = pd.read_csv("/workspaces/2026SE_MLOPSTask_Michael.L/wca_export/csv/WCA_export_competitions.csv")
competitions = competitions.drop(
    columns=[
        "information",
        "external_website",
        "venue",
        "venue_address",
        "venue_details",
        "cell_name",
        "event_specs",
        "delegates",
        "organizers",
        "latitude_microdegrees",
        "longitude_microdegrees",
    ]
)

#### Dealing with null values

Null values during data analysis can cause runtime errors and unexpected results. It is important to identify null values and deal with them appropriately before training a model.

The `isnull().sum()` method counts the number of missing values in each column of a DataFrame. This is an essential early step in data preprocessing because null values can cause errors or distort model training. By identifying which columns contain missing data, we can decide whether to remove affected rows/columns or replace the missing values with an appropriate substitute.

In [3]:
results.isnull().sum()

pos                  0
best                 0
average              0
competition_id       0
event_id             0
person_name          0
person_id            0
person_country_id    0
dtype: int64

In [4]:
persons.isnull().sum()

name            0
gender        398
wca_id          0
country_id      0
dtype: int64

In [5]:
competitions.isnull().sum()

id            0
name          0
city_name     0
country_id    0
cancelled     0
year          0
month         0
day           0
end_year      0
end_month     0
end_day       0
dtype: int64

After identifying missing values, they must be handled before training a model. Categorical null values are typically removed using `dropna()` because replacing them would introduce incorrect or arbitrary information. Numerical null values can be replaced using `fillna()`, often with the mean of the column, which preserves the dataset size while minimally affecting the distribution. This ensures the dataset is clean, consistent, and ready for modelling. In this example, gender was the only column to have null values, and since a gender can never really be determined or engineered from the existing features that we have, such as name or country, we must remove the rows using `dropna()`. 

In [6]:
# Remove Null values
persons = persons.dropna(subset=['gender'])
persons.isnull().sum()

name          0
gender        0
wca_id        0
country_id    0
dtype: int64

#### Remove Duplicates

The `drop_duplicates()` method removes repeated rows from the dataset based on one the selected column/s. This prevents duplicated samples from biasing the model or distorting statistical analysis. I specificaly removed the duplicates from the "average" column to ensure that identical performance values were not counted multiple time (the results dataset is also extemely big, and this cut down the size drastically, freeing up memory). Cleaning duplicates improves data quality, reduces noise, and ensures that each data point contributes fairly to the machine learning process.

In [7]:
results = results.drop_duplicates(subset=["average"])
results.head()

,pos,best,average,competition_id,event_id,person_name,person_id,person_country_id
0,15,1968,2128,LyonOpen2007,333,Etienne Amany,2007AMAN01,Cote d_Ivoire
1,16,1731,2140,LyonOpen2007,333,Thomas Rouault,2004ROUA01,France
2,17,2305,2637,LyonOpen2007,333,Antoine Simon-Chautemps,2005SIMO01,France
4,19,2677,2906,LyonOpen2007,333,Marlène Desmaisons,2007DESM01,France
5,20,1869,2910,LyonOpen2007,333,Ton Dennenbroek,2003DENN01,Netherlands


#### Remove outliers

Outliers can often distort statistical analysis and negatively impact machine learning models. The Interquartile Range (IQR) identifies outliers by calculating the bottom 25th percentile (Q1), 75th percentile (Q3), and the IQR (Q3 − Q1). Any value below Q1 − 1.5×IQR or above Q3 + 1.5×IQR is considered an outlier. These values are removed to produce a cleaner, more reliable dataset. Removing outliers improves model stability, reduces noise, and ensures that the model learns from the typical patterns rather than the extreme or anomalous data points.

note that 'average' is the only numerical value in the data that I will use

In [8]:
print(results['average'].describe())
Q1 = results['average'].quantile(0.25)
Q3 = results['average'].quantile(0.75)
IQR = Q3 - Q1
print(f'Outliers are a BP above {Q3 + IQR * 1.5} or below {Q1 - IQR * 1.5}')


count    15288.000000
mean      8717.795199
std       5796.756158
min        384.000000
25%       4230.750000
50%       8052.500000
75%      12030.250000
max      52507.000000
Name: average, dtype: float64
Outliers are a BP above 23729.5 or below -7468.5


In [9]:
# Filter averages within the acceptable range
results = results[(results['average'] >= Q1 - 1.5 * IQR) & (results['average'] <= Q3 + 1.5 * IQR)]
print(results['average'].describe())
results.head()

count    15015.000000
mean      8342.874259
std       5080.664344
min        384.000000
25%       4162.500000
50%       7916.000000
75%      11793.500000
max      23729.000000
Name: average, dtype: float64


,pos,best,average,competition_id,event_id,person_name,person_id,person_country_id
0,15,1968,2128,LyonOpen2007,333,Etienne Amany,2007AMAN01,Cote d_Ivoire
1,16,1731,2140,LyonOpen2007,333,Thomas Rouault,2004ROUA01,France
2,17,2305,2637,LyonOpen2007,333,Antoine Simon-Chautemps,2005SIMO01,France
4,19,2677,2906,LyonOpen2007,333,Marlène Desmaisons,2007DESM01,France
5,20,1869,2910,LyonOpen2007,333,Ton Dennenbroek,2003DENN01,Netherlands


#### Scaling features to a common range

Feature scaling transforms numerical values into a consistent range so that no single feature dominates the model due to its size/magnitude. In this model, I scaled the ‘average’ column using min–max, which maps all of the values into a range of 0–1. This improves model stability, prevents large-scale features from overpowering smaller ones, and ensures that distance-based or gradient-based algorithms behave correctly. The minimum and maximum values used for scaling must be saved so that new data can be scaled consistently during prediction.

In [10]:
scale_feature = 'average'

#the minimum value with space for outliers
MIN_AVG = 200

#the maximum value with space for outliers
MAX_AVG = 23750

#scale features
results[scale_feature] = [(X - MIN_AVG) / (MAX_AVG - MIN_AVG) for X in results[scale_feature]]

results.describe()

,pos,best,average
count,15015.000000,15015.000000,15015.000000
mean,57.294572,6494.918415,0.345770
std,58.372608,3789.183806,0.215739
min,1.000000,324.000000,0.007813
25%,22.000000,3410.500000,0.168259
50%,43.000000,6250.000000,0.327643
75%,75.000000,9065.000000,0.492293
max,908.000000,21561.000000,0.999108


> [!important]
> You need to save the calculations for each dataset you scale for scaling new values for prediction. Use [2.1.2.data.records.md](2.1.2.data.records.md) to record this information.

#### Save the wrangled data to CSV

In [11]:
results.to_csv('../2.2.Feature_Engineering/2.2.1.wrangled_results.csv', index=False)
persons.to_csv("../2.2.Feature_Engineering/2.2.1.wrangled_persons.csv", index=False)
countries.to_csv("../2.2.Feature_Engineering/2.2.1.wrangled_countries.csv", index=False)
competitions.to_csv("../2.2.Feature_Engineering/2.2.1.wrangled_competitions.csv", index=False)
results_all_events.to_csv("../2.2.Feature_Engineering/2.2.1.unwrangled_results.csv", index=False)